# FitCheck 가상 피팅 데모 (IDM-VTON 실험 버전)

기존 FASHN 대신 **IDM-VTON (SDXL 기반 고화질)** 엔진을 연동하여 테스트하는 노트북입니다.
사이즈 추천 및 자세 검사 기능은 기존과 동일하게 동작합니다.

**필수 확인: 상단 메뉴 `런타임` → `런타임 유형 변경` → `GPU (T4)` 로 설정되어 있는지 확인하세요.**

셀을 위에서부터 차례로 실행하면 마지막 셀에 **`https://....gradio.live`** 공개 링크가 생성됩니다.

In [ ]:
# 1. GPU 사양 확인 (T4 16GB 권장)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 코드 및 저장소 내려받기
import os

APP = '/content/vfa'
SCRATCH = '/content/scratch'
IDM_REPO = os.path.join(SCRATCH, 'IDM-VTON')

os.makedirs(SCRATCH, exist_ok=True)

# 우리 앱 저장소 받기
if not os.path.exists(APP):
    !git clone -q https://github.com/jhjung03eee/virtual-fitting-app.git {APP}
else:
    !git -C {APP} pull -q

# IDM-VTON 공식 저장소 받기
if not os.path.exists(IDM_REPO):
    !git clone -q https://github.com/yisol/IDM-VTON.git {IDM_REPO}

print('저장소 준비 완료')

In [ ]:
# 3. 의존성 패키지 설치 (약 2~3분)
#    (IDM-VTON 호환을 위해 diffusers==0.25.1 및 huggingface_hub<0.26.0 버전 필수)
!pip install -q diffusers==0.25.1 "huggingface_hub<0.26.0" transformers accelerate gradio mediapipe torchvision einops
print('패키지 설치 완료')

In [ ]:
# 4. IDM-VTON 연동 모듈 자동 배치
#    (깃허브에 아직 push하지 않았더라도 코랩에서 바로 동작하도록 자동 주입합니다)
import os

idm_core_code = '''"""IDM-VTON 추론 엔진 (app/idm_core.py)"""
import os, sys, time
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from paths import SCRATCH, REPO_ROOT

IDM_REPO = os.environ.get('IDM_REPO', os.path.join(SCRATCH, 'IDM-VTON'))
DEFAULT_STEPS = 30
DEFAULT_GUIDANCE = 2.0
DEFAULT_SEED = 42

CATEGORY_BY_CLOTH_TYPE = {
    'upper': 'upper_body', 'inner': 'upper_body', 'outer': 'upper_body',
    'lower': 'lower_body', 'overall': 'dresses',
}
CLOTH_TYPES = list(CATEGORY_BY_CLOTH_TYPE)
GARMENT_PHOTO_TYPES = ['flat-lay', 'model']
_pipeline, _preprocessors = None, None

def _to_rgb(x):
    from PIL import Image, ImageOps
    image = Image.open(x) if isinstance(x, str) else x
    return ImageOps.exif_transpose(image).convert('RGB')

def load_models():
    global _pipeline, _preprocessors
    if _pipeline is not None: return _pipeline, _preprocessors
    import torch
    from diffusers import AutoencoderKL
    if os.path.isdir(IDM_REPO) and IDM_REPO not in sys.path:
        sys.path.insert(0, IDM_REPO)
    from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
    from src.unet_hacked_tryon import UNet2DConditionModel
    from src.unet_hacked_garmnet import UNet2DConditionModel as GarmUNet

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model_id = 'yisol/IDM-VTON'
    print(f'[IDM-VTON] 가중치 로딩 중 ({model_id})...', flush=True)
    unet = UNet2DConditionModel.from_pretrained(model_id, subfolder='unet', torch_dtype=dtype)
    garm_net = GarmUNet.from_pretrained(model_id, subfolder='garm_net', torch_dtype=dtype)
    vae = AutoencoderKL.from_pretrained(model_id, subfolder='vae', torch_dtype=dtype)
    pipe = TryonPipeline.from_pretrained(model_id, unet=unet, garm_net=garm_net, vae=vae, torch_dtype=dtype)
    if torch.cuda.is_available():
        pipe.enable_model_cpu_offload()
        pipe.enable_vae_slicing()
        pipe.enable_vae_tiling()
    else:
        pipe.to(device)
    preprocessors = {}
    try:
        from preprocess.humanparsing.run_parsing import Parsing
        from preprocess.openpose.run_openpose import OpenPose
        preprocessors['parsing'] = Parsing(0) if torch.cuda.is_available() else Parsing(-1)
        preprocessors['openpose'] = OpenPose(0) if torch.cuda.is_available() else OpenPose(-1)
    except Exception as e:
        print(f'[IDM-VTON] 전처리기 안내: {e}')
    _pipeline, _preprocessors = pipe, preprocessors
    print('[IDM-VTON] 로딩 완료!', flush=True)
    return _pipeline, _preprocessors

def _preprocess_inputs(person_img, cloth_category, preprocessors):
    from PIL import Image
    import numpy as np
    width, height = 768, 1024
    person_img = person_img.resize((width, height), Image.Resampling.LANCZOS)
    if preprocessors and 'parsing' in preprocessors and 'openpose' in preprocessors:
        try:
            parsed_image, mask = preprocessors['parsing'](person_img)
            pose_image = preprocessors['openpose'](person_img)
            return person_img, mask, pose_image
        except Exception as e:
            print(f'[IDM-VTON] 전처리 폴백: {e}')
    mask = Image.new('L', (width, height), 0)
    mask_arr = np.array(mask)
    if cloth_category == 'lower_body':
        mask_arr[int(height * 0.45):int(height * 0.9), int(width * 0.15):int(width * 0.85)] = 255
    else:
        mask_arr[int(height * 0.15):int(height * 0.6), int(width * 0.15):int(width * 0.85)] = 255
    return person_img, Image.fromarray(mask_arr), person_img.copy()

def try_on(person, garment, cloth_type='upper', steps=DEFAULT_STEPS, guidance_scale=DEFAULT_GUIDANCE,
           seed=DEFAULT_SEED, garment_photo_type='flat-lay', garment_des='a clothing item', return_timing=False):
    import torch
    from PIL import Image
    if seed == -1: import random; seed = random.randint(0, 2**31 - 1)
    pipe, preprocessors = load_models()
    start = time.perf_counter()
    person_img = _to_rgb(person)
    garment_img = _to_rgb(garment).resize((768, 1024), Image.Resampling.LANCZOS)
    category = CATEGORY_BY_CLOTH_TYPE[cloth_type]
    person_img, mask, pose_img = _preprocess_inputs(person_img, category, preprocessors)
    generator = torch.Generator('cuda' if torch.cuda.is_available() else 'cpu').manual_seed(int(seed))
    with torch.no_grad():
        output = pipe(
            prompt=f'model is wearing {garment_des}',
            negative_prompt='monochrome, lowres, bad anatomy, worst quality, low quality',
            image=person_img, mask_image=mask, pose_img=pose_img, garm_img=garment_img,
            num_inference_steps=int(steps), guidance_scale=float(guidance_scale),
            generator=generator, height=1024, width=768,
        )
    result = output.images[0]
    timing = {'total_s': round(time.perf_counter() - start, 2), 'seed': int(seed), 'dtype': 'fp16' if torch.cuda.is_available() else 'fp32'}
    return (result, timing) if return_timing else result
'''

# gradio_app_idm.py 생성
gradio_app_idm_code = '''"""IDM-VTON 데모 앱"""
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import gradio as gr
import gradio_client.utils as _gcu
_orig = _gcu._json_schema_to_python_type
_gcu._json_schema_to_python_type = lambda s, d=None: 'Any' if isinstance(s, bool) else _orig(s, d)
_orig_get = _gcu.get_type
_gcu.get_type = lambda s: 'Any' if not isinstance(s, dict) else _orig_get(s)

from idm_core import try_on, load_models, DEFAULT_STEPS, DEFAULT_GUIDANCE, DEFAULT_SEED
from body_profile import BodyProfile
from photo_check import check_photo
from size_fit import SizeChart, recommend
import json

CLOTH_LABELS = {'상의': 'upper', '하의': 'lower', '원피스 (상하 일체)': 'overall', '이너': 'inner', '아우터': 'outer'}
CHART_DIR = os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), 'data', 'size_charts')

def load_chart(f): return SizeChart.from_dict(json.load(open(os.path.join(CHART_DIR, f), encoding='utf-8')))
def list_charts(cat=None):
    if not os.path.isdir(CHART_DIR): return []
    files = sorted(f for f in os.listdir(CHART_DIR) if f.endswith('.json'))
    return files if cat is None else [f for f in files if getattr(load_chart(f), 'category', None) == cat]
def charts_for_cloth(c): files = list_charts('lower' if CLOTH_LABELS.get(c) == 'lower' else 'upper'); return gr.update(choices=files, value=files[0] if files else None)

def size_advice(f, h, s, c, w, hip):
    if not f: return '치수표가 없어 사이즈 추천을 건너뜁니다.'
    try: prof = BodyProfile.from_inputs(height_cm=h, shoulder_cm=s, chest_circumference_cm=c, waist_circumference_cm=w, hip_circumference_cm=hip)
    except ValueError as e: return f'**사이즈 추천 불가** — {e}'
    probs = prof.validate()
    if probs: return '**입력값을 확인해주세요**\n\n' + '\n'.join(f'- {p}' for p in probs)
    body = prof.to_chart_dimensions()
    if not body: return '신체 치수를 입력하면 사이즈를 추천합니다.'
    chart = load_chart(f)
    rec = recommend(chart, body)
    if rec.best is None: return '**사이즈 추천 불가** — ' + ' '.join(rec.notes)
    lines = [f'### {rec.message}', '', f'**{chart.name}** 기준', '', '| 사이즈 | 항목별 여유분 | 입을 수 있음 |', '|---|---|---|']
    for fit in rec.ranked:
        det = ' / '.join(f'{d.dimension} {d.ease_cm:+.1f}cm ({d.label})' for d in fit.dimensions)
        lines.append(f'| {fit.size} | {det} | {"O" if fit.wearable else "X"} |')
    return '\n'.join(lines)

GARMENT_PHOTO_LABELS = {'상품 사진 (옷만 펴 놓고 찍음)': 'flat-lay', '착용 사진 (사람이 입고 찍음)': 'model'}

def photo_status(p):
    if p is None: return ''
    chk = check_photo(p)
    if chk.ok and not chk.warnings: return '✅ ' + chk.message()
    return ('❌ ' if chk.errors else '⚠️ ') + chk.message()

def run(p, g, c_lbl, gp_lbl, ign, st, gui, sd, f, h, s, c, w, hip):
    if p is None or g is None: raise gr.Error('인물과 옷 사진을 모두 올려주세요.')
    if not ign and not check_photo(p).ok: raise gr.Error(check_photo(p).message())
    adv = size_advice(f, h, s, c, w, hip)
    res, tim = try_on(p, g, cloth_type=CLOTH_LABELS[c_lbl], steps=int(st), guidance_scale=float(gui), seed=int(sd), return_timing=True)
    return res, adv, f"합성 {tim['total_s']:.0f}초 · 시드 {tim['seed']} · {tim['dtype']} (엔진: IDM-VTON)"

with gr.Blocks(title='사이즈 반영 가상 피팅 (IDM-VTON)') as demo:
    gr.Markdown('# 가상 피팅 데모 (IDM-VTON 실험 버전)\n인물 사진과 옷 사진을 올리면 고화질 SDXL 피팅이 진행됩니다.')
    with gr.Row():
        with gr.Column():
            person_in = gr.Image(label='인물 사진', type='pil', height=400)
            photo_status_out = gr.Markdown()
            garment_in = gr.Image(label='옷 사진', type='pil', height=400)
            cloth_in = gr.Radio(choices=list(CLOTH_LABELS.keys()), value='상의', label='옷 종류')
            garment_photo_in = gr.Radio(choices=list(GARMENT_PHOTO_LABELS.keys()), value='상품 사진 (옷만 펴 놓고 찍음)', label='옷 사진 종류')
            with gr.Accordion('내 신체 치수', open=True):
                height_in = gr.Number(value=175, label='키 (cm)')
                shoulder_in = gr.Number(value=None, label='어깨너비 (cm)')
                chest_in = gr.Number(value=None, label='가슴둘레 (cm)')
                waist_in = gr.Number(value=None, label='허리둘레 (cm)')
                hip_in = gr.Number(value=None, label='엉덩이둘레 (cm)')
                init = list_charts('upper')
                chart_in = gr.Dropdown(choices=init, value=init[0] if init else None, label='옷 치수표')
            with gr.Accordion('고급 설정', open=False):
                ignore_check_in = gr.Checkbox(value=False, label='사진 검사 무시하고 합성')
                steps_in = gr.Slider(20, 50, value=DEFAULT_STEPS, step=1, label='추론 스텝')
                guidance_in = gr.Slider(1.0, 4.0, value=DEFAULT_GUIDANCE, step=0.1, label='guidance scale')
                seed_in = gr.Number(value=DEFAULT_SEED, precision=0, label='시드')
            run_btn = gr.Button('피팅 해보기', variant='primary')
        with gr.Column():
            result_out = gr.Image(label='합성 결과', height=520)
            info_out = gr.Markdown()
            size_out = gr.Markdown(label='사이즈 추천')
    cloth_in.change(fn=charts_for_cloth, inputs=[cloth_in], outputs=[chart_in])
    person_in.change(fn=photo_status, inputs=[person_in], outputs=[photo_status_out])
    run_btn.click(fn=run, inputs=[person_in, garment_in, cloth_in, garment_photo_in, ignore_check_in, steps_in, guidance_in, seed_in, chart_in, height_in, shoulder_in, chest_in, waist_in, hip_in], outputs=[result_out, size_out, info_out])

if __name__ == '__main__':
    load_models()
    options = dict(share=True, show_error=True, prevent_thread_lock=True)
    try: _, l_url, s_url = demo.queue().launch(show_api=False, **options)
    except TypeError: _, l_url, s_url = demo.queue().launch(**options)
    print('=' * 60)
    print('공개 링크 :', s_url or '(생성 실패)')
    print('=' * 60)
    demo.block_thread()
'''

with open(os.path.join(APP, 'app', 'idm_core.py'), 'w', encoding='utf-8') as f:
    f.write(idm_core_code)
with open(os.path.join(APP, 'app', 'gradio_app_idm.py'), 'w', encoding='utf-8') as f:
    f.write(gradio_app_idm_code)

print('IDM-VTON 파일 배치 완료!')

In [ ]:
# 5. IDM-VTON 데모 앱 실행 (출력창 하단의 *.gradio.live 링크로 접속)
#    (최초 실행 시 허깅페이스에서 가중치를 다운로드하므로 3~5분 정도 소요됩니다)
os.environ['VFA_SCRATCH'] = SCRATCH
os.environ['IDM_REPO'] = IDM_REPO

!python {APP}/app/gradio_app_idm.py